# QuEEN-DTA: run the benchmark

Trains QuEEN-DTA on Davis and KIBA with a fixed test fold and 5 cross-validation runs per dataset, then writes the scores and charts.

Run the cells in order. Step 1 creates `.venv` in the repository and installs everything there; nothing is installed into the Python that runs this notebook.

In [ ]:
# Step 1: create the repository's own virtual environment (.venv) and install everything in it.
# This is the install list: edit PACKAGES / TORCH_INDEX_URL here if you need different versions.
# Nothing is installed into the Python that runs this notebook.
import hashlib
import json
import os
import subprocess
import sys
import venv
from pathlib import Path

TORCH_PACKAGES = ["torch"]
TORCH_INDEX_URL = "https://download.pytorch.org/whl/cu126"
PACKAGES = [
    "biopython",
    "matplotlib",
    "numpy",
    "pandas",
    "pennylane",
    "psutil",
    "rdkit",
    "scikit-learn",
]

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts" / "run_benchmark.py").exists():
    ROOT = ROOT.parent  # notebook opened from notebooks/
if not (ROOT / "scripts" / "run_benchmark.py").exists():
    raise RuntimeError(
        f"Open this notebook from the repository (scripts/run_benchmark.py not found near {Path.cwd()})"
    )
if sys.version_info < (3, 10):
    raise RuntimeError("Python 3.10 or newer is needed (pennylane)")

VENV_DIR = ROOT / ".venv"
PY = VENV_DIR / ("Scripts/python.exe" if os.name == "nt" else "bin/python")


def stream(command):
    process = subprocess.Popen(
        [str(c) for c in command],
        cwd=ROOT,
        env={**os.environ, "PYTHONPATH": str(ROOT / "src")},
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    code = process.wait()
    if code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {command}")


if not PY.exists():
    print(f"Creating virtual environment in {VENV_DIR}")
    venv.EnvBuilder(with_pip=True).create(VENV_DIR)

marker = VENV_DIR / ".packages_installed"
stamp = hashlib.sha256(
    json.dumps([TORCH_PACKAGES, TORCH_INDEX_URL, PACKAGES]).encode()
).hexdigest()
if not marker.exists() or marker.read_text() != stamp:
    stream([PY, "-m", "pip", "install", "--upgrade", "pip"])
    stream([PY, "-m", "pip", "install", *TORCH_PACKAGES, "--index-url", TORCH_INDEX_URL])
    stream([PY, "-m", "pip", "install", *PACKAGES])
    marker.write_text(stamp)
print("Virtual environment ready:", PY)

In [ ]:
# Step 2: check that the virtual environment sees the GPU and the data is intact.
stream([PY, "-c", (
    "import torch;"
    "assert torch.cuda.is_available(), 'No CUDA GPU: update the NVIDIA driver or change TORCH_INDEX_URL';"
    "p = torch.cuda.get_device_properties(0);"
    "print('GPU:', p.name, round(p.total_memory / 2**30, 1), 'GB | torch', torch.__version__)"
)])
stream([PY, "-c", (
    "from queen_dta.dataset import load_dataset, check_ids;"
    "[print(d, len(load_dataset(d)[0]), 'pairs, folds OK') for d in ('davis', 'kiba')]"
)])

In [ ]:
# Step 3: train QuEEN-DTA (200 epochs per run, 5 runs per dataset).
# Finished runs are saved in results/ and skipped when you run this cell again.
# For a quick test use DATASETS = ["davis"] and FOLDS = [0].
DATASETS = ["davis", "kiba"]
FOLDS = [0, 1, 2, 3, 4]

stream([PY, "-u", "scripts/run_benchmark.py", "--datasets", *DATASETS, "--folds", *[str(f) for f in FOLDS]])

In [ ]:
# Step 4: scores (saved in results/*.csv) and the comparison charts (saved in assets/figures/).
stream([PY, "scripts/summarize_results.py"])
stream([PY, "scripts/plot_comparison.py"])